# BAAI/bge-m3 Embedding Generation

This notebook generates dense embeddings for all WikiCS nodes using the [BAAI/bge-m3](https://huggingface.co/BAAI/bge-m3) model.

**Features:** Incremental saving — progress is saved after every batch and already-embedded nodes are skipped on restart.

**Output:** `master_embeddings.parquet` with columns `id` (node id) and `embedding` (dense vector).

In [ ]:
import json
import os
import torch
import pandas as pd
from FlagEmbedding import BGEM3FlagModel

In [ ]:
# Load the dataset
with open("../../data/data-embeddings.json", "r") as f:
    data = json.load(f)

nodes = data["nodes"]
print(f"Total nodes: {len(nodes)}")

In [ ]:
# Load existing progress if available
OUTPUT_FILE = "master_embeddings.parquet"

if os.path.exists(OUTPUT_FILE):
    df_existing = pd.read_parquet(OUTPUT_FILE)
    done_ids = set(df_existing["id"].tolist())
    print(f"Resuming: {len(done_ids)} nodes already embedded, {len(nodes) - len(done_ids)} remaining.")
else:
    df_existing = pd.DataFrame(columns=["id", "embedding"])
    done_ids = set()
    print("Starting fresh — no existing embeddings found.")

# Filter to only the nodes that still need embedding
remaining_nodes = [n for n in nodes if n["id"] not in done_ids]
print(f"Nodes to embed: {len(remaining_nodes)}")

In [ ]:
# Load the BAAI/bge-m3 model
# use_fp16=True to fit within 8GB VRAM
model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True, device="cuda")
print("Model loaded successfully.")

In [ ]:
# Generate embeddings in batches with incremental saving
BATCH_SIZE = 32

remaining_ids = [n["id"] for n in remaining_nodes]
remaining_texts = [n["raw_text"] for n in remaining_nodes]
total_batches = (len(remaining_texts) + BATCH_SIZE - 1) // BATCH_SIZE

rows = df_existing.to_dict("records")

for i in range(0, len(remaining_texts), BATCH_SIZE):
    batch_num = i // BATCH_SIZE + 1
    batch_texts = remaining_texts[i : i + BATCH_SIZE]
    batch_ids = remaining_ids[i : i + BATCH_SIZE]

    output = model.encode(
        batch_texts,
        batch_size=BATCH_SIZE,
        max_length=8192,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False,
    )

    for node_id, emb in zip(batch_ids, output["dense_vecs"]):
        rows.append({"id": node_id, "embedding": emb.tolist()})

    # Save incrementally after each batch
    df_save = pd.DataFrame(rows)
    df_save.to_parquet(OUTPUT_FILE, index=False)

    print(f"Batch {batch_num}/{total_batches} done — {len(rows)} total embeddings saved.")

print(f"\nFinished! {len(rows)} embeddings saved to {OUTPUT_FILE}")

In [ ]:
# Verify the saved file
df_check = pd.read_parquet(OUTPUT_FILE)
print(f"Verified: {len(df_check)} rows, embedding dim = {len(df_check.iloc[0]['embedding'])}")
df_check.head()